# EDA: Análisis de Valores 0 en el MDT en Zonas Costeras (Efecto MAUP)

En este notebook vamos a explorar y justificar estadísticamente por qué los hexágonos H3 situados en primera línea de costa presentan valores anómalos de `elevation_mean` (muy cercanos a 0 o negativos) y `slope_mean` (pendientes casi perfectas de 0 grados).

Demostraremos que esto no es un error de los datos de tierra firme, sino un efecto de dilución (Modifiable Areal Unit Problem) causado porque el raster MDT incluye el mar codificado como `0` en lugar de `NoData`.


In [1]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

user = os.getenv("AZURE_DB_USER")
pw   = os.getenv("AZURE_DB_PASSWORD")
host = os.getenv("AZURE_DB_HOST")
port = os.getenv("AZURE_DB_PORT", "5432")
db   = os.getenv("AZURE_DB_NAME")

connection_string = f"postgresql://{user}:{pw}@{host}:{port}/{db}"
engine = create_engine(connection_string)


### 1. Extraer Hexágonos Problemáticos de la Base de Datos

Vamos a buscar aquellos hexágonos donde la elevación media sea menor o igual a 1 metro, o que tengan pendientes medias sospechosamente bajas (casi 0).


In [3]:
query = """
SELECT 
    m.h3_index,
    m.elevation_mean,
    m.elevation_min,
    m.slope_mean,
    h.geometry
FROM bronze.bronze_mdt_stats m
JOIN silver.silver_h3_grid h ON m.h3_index = h.h3_index
WHERE m.elevation_mean <= 1 OR m.slope_mean < 0.05
ORDER BY m.elevation_mean ASC;
"""

# Cargamos directamente como GeoDataFrame
gdf = gpd.read_postgis(query, engine, geom_col='geometry')
print(f"Se han encontrado {len(gdf)} hexágonos anómalos.")
gdf.head(10)


Se han encontrado 69 hexágonos anómalos.


,h3_index,elevation_mean,elevation_min,slope_mean,geometry
0,88344c51cbfffff,-0.067786,-3.899,0.374430,"POLYGON ((-16.62516 28.01701, -16.6302 28.0143..."
1,88344cc239fffff,-0.057085,-2.282,0.077262,"POLYGON ((-16.35806 28.30645, -16.36311 28.303..."
2,88346a6e59fffff,-0.046227,-1.016,0.045205,"POLYGON ((-16.11495 28.55647, -16.12001 28.553..."
3,8834412235fffff,-0.018061,-0.966,0.107270,"POLYGON ((-16.86848 28.29117, -16.87353 28.288..."
4,88344ccd5bfffff,-0.015277,-2.048,0.126399,"POLYGON ((-16.46404 28.11201, -16.46908 28.109..."
5,883441a4b5fffff,-0.013512,-0.896,0.027177,"POLYGON ((-16.47936 28.45936, -16.48442 28.456..."
6,883441a4d9fffff,0.000000,0.000,0.012306,"POLYGON ((-16.43382 28.49861, -16.43889 28.495..."
7,88346a6d53fffff,0.000504,-0.162,0.024692,"POLYGON ((-16.22136 28.48576, -16.22642 28.483..."
8,88344cc247fffff,0.000832,-2.528,0.303298,"POLYGON ((-16.35804 28.33734, -16.36309 28.334..."
9,88346a61d9fffff,0.002252,-0.230,0.060731,"POLYGON ((-16.15019 28.61018, -16.15526 28.607..."


Como podemos observar en la tabla anterior, tenemos hexágonos con medias de altitud negativas (`-0.06m`) y pendientes minúsculas (`0.02°`). Esto es físicamente imposible para la orografía escarpada de Tenerife, incluso a pie de playa.


In [4]:
# Estadísticos descriptivos de los hexágonos "cota cero"
gdf[['elevation_mean', 'elevation_min', 'slope_mean']].describe()

,elevation_mean,elevation_min,slope_mean
count,69.000000,69.000000,69.000000
mean,0.275325,-1.141768,0.469784
std,0.284641,1.056371,0.379051
min,-0.067786,-4.618000,0.008930
25%,0.021226,-1.585000,0.188830
50%,0.188664,-0.902000,0.380865
75%,0.467534,-0.337000,0.635461
max,0.922747,0.000000,1.632747


### 2. Visualización en el Mapa (Prueba de Costa)

Para confirmar nuestra hipótesis de que estos píxeles "0" provienen del mar, vamos a pintarlos en un mapa interactivo.
Si nuestra teoría es cierta, todos estos hexágonos anómalos deberían formar un anillo alrededor de la costa, evidenciando que el mar arrastra las medias a la baja al promediar tierra y agua en un mismo hexágono H3.


In [6]:
# Crear mapa centrado en Tenerife
m = folium.Map(location=[28.291565, -16.629129], zoom_start=10, tiles='CartoDB Positron')

# Añadir los hexágonos anómalos en color rojo
folium.GeoJson(
    gdf,
    style_function=lambda feature: {
        'fillColor': 'red',
        'color': 'darkred',
        'weight': 1,
        'fillOpacity': 0.6,
    },
    tooltip=folium.GeoJsonTooltip(fields=['h3_index', 'elevation_mean', 'slope_mean'])
).add_to(m)

m.save("mapa_mdt_costa.html")
print("Mapa guardado como mapa_mdt_costa.html")


Mapa guardado como mapa_mdt_costa.html


### Conclusión y Fix en la Ingesta

El mapa interactivo confirma al 100% que **el problema se concentra exclusivamente en la franja costera**.

**Causa Raíz:** 
El algoritmo de estadística zonal promedia los píxeles que caen dentro del polígono del hexágono. Un hexágono en la costa puede tener un 80% de área en el mar (cota 0, pendiente 0) y un 20% en tierra firme (cota 10m, pendiente 5º). Al hacer la media ponderada por área, el resultado será `(0.8 * 0) + (0.2 * 10) = 2m` de elevación y `(0.8 * 0) + (0.2 * 5) = 1º` de pendiente. 

**Solución aplicada:**
En el script de ingesta de python (`02_ingest_mdt_to_postgres.py`), hemos añadido una máscara para convertir a `np.nan` todos los valores del raster `elevation <= 0.0` antes de pasar el array a la librería `rasterstats`. Al ser `NaN`, la librería simplemente ignorará los píxeles marinos y calculará la media **solamente con el 20% de píxeles que sí son de tierra firme**, capturando la altitud y pendiente real de la costa de Tenerife.
